# Lab | Web Scraping - Solution Notebook

**Website**: Books to Scrape (http://books.toscrape.com/)

**Goal**: Build a function `scrape_books(min_rating, max_price)` that scrapes book data and returns a clean pandas DataFrame.

## How Web Scraping Works

Web scraping has 3 main steps:
1. **Request** — send a request to the website and get the HTML back
2. **Parse** — use BeautifulSoup to read and navigate the HTML
3. **Extract** — find the specific elements you want and pull out the data

Think of HTML as a tree of nested boxes. BeautifulSoup helps you navigate and find specific boxes.

## Setup - Import Libraries

In [1]:
import requests                          # sends HTTP requests to websites
from bs4 import BeautifulSoup           # parses and navigates HTML
import pandas as pd                      # stores and structures our data
import time                              # adds delays between requests (polite scraping)

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 1: Understanding the Website Structure

Before writing the scraper, let's explore the website to understand its structure.

In [2]:
# The base URL of the website
BASE_URL = "http://books.toscrape.com/"

# Send a request to the website
# requests.get() fetches the page — like opening a browser and going to a URL
response = requests.get(BASE_URL)

# Check the status code
# 200 = success, 404 = not found, 403 = forbidden
print("Status code:", response.status_code)
print("Success!" if response.status_code == 200 else "Something went wrong!")

Status code: 200
Success!


In [3]:
# Parse the HTML using BeautifulSoup
# 'html.parser' tells BeautifulSoup to use Python's built-in HTML parser
soup = BeautifulSoup(response.content, 'html.parser')

# Find all book elements on the page
# Each book is inside an <article> tag with class 'product_pod'
books = soup.find_all('article', class_='product_pod')

print(f"Number of books on the first page: {len(books)}")

Number of books on the first page: 20


In [4]:
# Let's look at a single book's HTML to understand its structure
# This helps us know which tags and attributes to target
first_book = books[0]
print(first_book.prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



## Step 2: Helper Functions

We need two helper functions:
1. One to convert word ratings ('One', 'Two'...) to numbers
2. One to get detailed info from each book's individual page

In [5]:
# Helper Function 1: Convert word rating to number
# On the website, ratings are stored as words: 'One', 'Two', 'Three', 'Four', 'Five'
# We need to convert these to numbers: 1, 2, 3, 4, 5

def word_to_rating(word):
    """
    Converts a word rating to a number.
    Example: 'Four' → 4
    """
    rating_map = {
        'One': 1,
        'Two': 2,
        'Three': 3,
        'Four': 4,
        'Five': 5
    }
    return rating_map.get(word, 0)  # Returns 0 if the word is not found

# Test it
print(word_to_rating('Four'))  # Should print 4
print(word_to_rating('One'))   # Should print 1

4
1


In [6]:
# Helper Function 2: Get detailed book information from a book's individual page
# The main listing page only shows basic info (title, price, rating)
# To get UPC, Genre, Availability, and Description we need to visit each book's page

def get_book_details(book_url):
    """
    Visits a book's individual page and extracts:
    - UPC
    - Genre
    - Availability
    - Description
    """
    # Send request to the book's page
    response = requests.get(book_url)
    
    # Check if request was successful
    if response.status_code != 200:
        return None
    
    # Parse the book page HTML
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Extract UPC from the product information table
    # The table has rows with 'th' (header) and 'td' (data) elements
    table = soup.find('table', class_='table table-striped')
    rows = table.find_all('tr') if table else []
    
    upc = ''
    availability = ''
    
    for row in rows:
        header = row.find('th')
        value = row.find('td')
        if header and value:
            if header.text.strip() == 'UPC':
                upc = value.text.strip()
            elif header.text.strip() == 'Availability':
                availability = value.text.strip()
    
    # Extract Genre from breadcrumb navigation
    # Breadcrumb looks like: Home > Category > Genre > Book Title
    breadcrumb = soup.find('ul', class_='breadcrumb')
    genre = ''
    if breadcrumb:
        breadcrumb_items = breadcrumb.find_all('li')
        # Genre is the 3rd item (index 2): Home > Books > Genre > Title
        if len(breadcrumb_items) >= 3:
            genre = breadcrumb_items[2].text.strip()
    
    # Extract Description
    # Description is in a <p> tag after the <div id='product_description'>
    description = ''
    desc_div = soup.find('div', id='product_description')
    if desc_div:
        desc_p = desc_div.find_next_sibling('p')
        if desc_p:
            description = desc_p.text.strip()
    
    return {
        'upc': upc,
        'availability': availability,
        'genre': genre,
        'description': description
    }

print("Helper functions defined!")

Helper functions defined!


## Step 3: The Main Scraping Function

Now we build the `scrape_books` function that:
1. Goes through all pages of the website
2. Extracts basic info from the listing page
3. Visits each book's page for detailed info
4. Filters by min_rating and max_price
5. Returns a clean pandas DataFrame

In [7]:
def scrape_books(min_rating, max_price):
    """
    Scrapes book data from Books to Scrape website.
    
    Parameters:
    -----------
    min_rating : int or float
        Minimum star rating (1-5). Books below this rating are excluded.
    max_price : float
        Maximum price in pounds. Books above this price are excluded.
    
    Returns:
    --------
    pandas DataFrame with columns:
        UPC, Title, Price (£), Rating, Genre, Availability, Description
    """
    
    BASE_URL = "http://books.toscrape.com/"
    all_books = []       # List to collect all book data
    page_number = 1      # Start from page 1
    
    print(f"Starting scrape: min_rating={min_rating}, max_price=£{max_price}")
    print("Scraping pages...")
    
    # Loop through all pages until there are no more pages
    while True:
        
        # Build the URL for the current page
        # Page 1: catalogue/page-1.html, Page 2: catalogue/page-2.html, etc.
        if page_number == 1:
            url = BASE_URL
        else:
            url = f"{BASE_URL}catalogue/page-{page_number}.html"
        
        # Send request to the page
        response = requests.get(url)
        
        # If page doesn't exist, we've reached the end — stop the loop
        if response.status_code != 200:
            print(f"No more pages after page {page_number - 1}. Done!")
            break
        
        print(f"  Scraping page {page_number}...", end=" ")
        
        # Parse the page HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find all book cards on this page
        books = soup.find_all('article', class_='product_pod')
        
        # If no books found, stop
        if not books:
            break
        
        # Process each book on the page
        for book in books:
            
            # --- Extract Title ---
            # Title is in an <h3> tag inside an <a> tag
            # The 'title' attribute has the full title (not truncated)
            title_tag = book.find('h3').find('a')
            title = title_tag['title'] if title_tag else 'N/A'
            
            # --- Extract Price ---
            # Price is in a <p> tag with class 'price_color'
            # We remove the '£' symbol and convert to float
            price_tag = book.find('p', class_='price_color')
            price_text = price_tag.text.strip() if price_tag else '0'
            price = float(price_text.replace('£', '').replace('Â', '').strip())
            
            # --- Extract Rating ---
            # Rating is stored as a CSS class in the <p> tag
            # Example: <p class="star-rating Four"> means 4 stars
            rating_tag = book.find('p', class_='star-rating')
            rating_word = rating_tag['class'][1] if rating_tag else 'Zero'
            rating = word_to_rating(rating_word)
            
            # --- Apply Filters ---
            # Skip this book if it doesn't meet our criteria
            if rating < min_rating or price > max_price:
                continue
            
            # --- Get Book Detail Page URL ---
            # The link to the book's page is in the <a> tag inside <h3>
            book_link = title_tag['href'] if title_tag else ''
            
            # Fix the URL — the href is relative, we need to make it absolute
            if '../../../' in book_link:
                book_url = BASE_URL + 'catalogue/' + book_link.replace('../../../', '')
            elif '../../' in book_link:
                book_url = BASE_URL + 'catalogue/' + book_link.replace('../../', '')
            else:
                book_url = BASE_URL + 'catalogue/' + book_link
            
            # --- Get Detailed Info from Book's Page ---
            details = get_book_details(book_url)
            
            if details:
                all_books.append({
                    'UPC': details['upc'],
                    'Title': title,
                    'Price (£)': price,
                    'Rating': rating,
                    'Genre': details['genre'],
                    'Availability': details['availability'],
                    'Description': details['description']
                })
            
            # Be polite — add a small delay between requests
            # This prevents overloading the server
            time.sleep(0.1)
        
        print(f"Found {len(books)} books. Total collected so far: {len(all_books)}")
        
        # Move to the next page
        page_number += 1
    
    # Convert the list of dictionaries to a pandas DataFrame
    df = pd.DataFrame(all_books)
    
    print(f"\nScraping complete!")
    print(f"Total books matching criteria: {len(df)}")
    
    return df

print("scrape_books function defined!")

scrape_books function defined!


## Step 4: Run the Scraper

Now we call the function with the required parameters:
- **min_rating = 4** (4 stars and above)
- **max_price = 20** (£20 and below)

In [8]:
# Run the scraper with the required parameters
# Note: This will take a few minutes as it visits each book's page
books_df = scrape_books(min_rating=4, max_price=20)
books_df.head(10)

Starting scrape: min_rating=4, max_price=£20
Scraping pages...
  Scraping page 1... Found 20 books. Total collected so far: 0
  Scraping page 2... Found 20 books. Total collected so far: 2
  Scraping page 3... Found 20 books. Total collected so far: 5
  Scraping page 4... Found 20 books. Total collected so far: 5
  Scraping page 5... Found 20 books. Total collected so far: 9
  Scraping page 6... Found 20 books. Total collected so far: 11
  Scraping page 7... Found 20 books. Total collected so far: 13
  Scraping page 8... Found 20 books. Total collected so far: 13
  Scraping page 9... Found 20 books. Total collected so far: 15
  Scraping page 10... Found 20 books. Total collected so far: 15
  Scraping page 11... Found 20 books. Total collected so far: 18
  Scraping page 12... Found 20 books. Total collected so far: 19
  Scraping page 13... Found 20 books. Total collected so far: 19
  Scraping page 14... Found 20 books. Total collected so far: 21
  Scraping page 15... Found 20 books. Tot

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
1,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
2,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
3,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."
4,709822d0b5bcb7f4,Thirst,17.27,5,Fiction,In stock (16 available),"On a searing summer Friday, Eddie Chapman has ..."
5,0fa6dceead7ce47a,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,Sequential Art,In stock (16 available),THE LONG-AWAITED STORY OF FANGIRLS TAKING ON T...
6,0e691eda369f4e09,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,Fantasy,In stock (16 available),Just as Annie and Liam are busy making plans t...
7,889139b8e9c4cb36,"Outcast, Vol. 1: A Darkness Surrounds Him (Out...",15.44,4,Sequential Art,In stock (16 available),NEW HORROR SERIES FROM THE WALKING DEAD CREATO...
8,72f9d5be3472d34e,Mama Tried: Traditional Italian Cooking for th...,14.02,4,Food and Drink,In stock (16 available),Cecilia Granata grew up cooking with her famil...
9,9e28048cea8d41f7,First and First (Five Boroughs #3),15.97,4,Romance,In stock (16 available),"Caleb Stone was raised on the Upper East Side,..."


## Step 5: Explore the Results

In [9]:
# Basic info about our scraped dataset
print("Shape:", books_df.shape)
print()
print("Data types:")
print(books_df.dtypes)
print()
print("Null values:")
print(books_df.isnull().sum())

Shape: (74, 7)

Data types:
UPC              object
Title            object
Price (£)       float64
Rating            int64
Genre            object
Availability     object
Description      object
dtype: object

Null values:
UPC             0
Title           0
Price (£)       0
Rating          0
Genre           0
Availability    0
Description     0
dtype: int64


In [10]:
# Distribution of ratings
print("Rating distribution:")
print(books_df['Rating'].value_counts().sort_index())

Rating distribution:
Rating
4    33
5    41
Name: count, dtype: int64


In [11]:
# Distribution of genres
print("Genre distribution:")
print(books_df['Genre'].value_counts())

Genre distribution:
Genre
Default               10
Sequential Art         9
Young Adult            6
Nonfiction             6
Fiction                5
Food and Drink         4
Poetry                 4
Fantasy                3
Romance                3
Womens Fiction         2
Childrens              2
Humor                  2
Philosophy             2
Add a comment          2
Science                2
Mystery                1
Adult Fiction          1
Science Fiction        1
Autobiography          1
Horror                 1
Spirituality           1
Classics               1
Christian Fiction      1
Art                    1
Erotica                1
Business               1
Historical Fiction     1
Name: count, dtype: int64


In [12]:
# Price statistics
print("Price statistics:")
print(books_df['Price (£)'].describe())

Price statistics:
count    74.000000
mean     14.531892
std       2.726005
min      10.000000
25%      12.340000
50%      14.415000
75%      16.677500
max      19.690000
Name: Price (£), dtype: float64


In [13]:
# Save to CSV for future use
books_df.to_csv('scraped_books.csv', index=False)
print("Data saved to 'scraped_books.csv'")

Data saved to 'scraped_books.csv'


## Test with Different Parameters

In [14]:
# Test with stricter filters — only 5-star books under £15
# Uncomment to run:
# top_books = scrape_books(min_rating=5, max_price=15)
# print(top_books.shape)
# top_books.head()